# Lab 4.4 · The datasheet's preparation section

Day 4 · Session 4 (Handing over prepared data) · 25 min · AI-tutor: an assistant may help with
wording, never with the numbers.

Hatoon will hand `complaints_prepared` to analysts who were not in the room this week. A datasheet
(Gebru et al.) answers their questions; its "collection, preprocessing and labelling" part is yours.
You write it from a run log, the rule results and the agreement report, then check mechanically that
every number in it appears in those sources.

**Reference state.** A pipeline run on the corrected September extract (made here, as in Lab 4.2);
`data/reference/agreement_r1_r2.csv` (the pilot's agreement by category and variety, rounds 1 and 2);
`pipeline/contract.odcs.yaml`.

**Tasks**
- [ ] 1 · What a user of the table needs to know (4')
- [ ] 2 · Fill the section from the run log and rule results (9')
- [ ] 3 · Agreement by variety, and the known limits (7')
- [ ] 4 · Every number traced to the run (5')
- [ ] ✱ Stretch · The contract and the datasheet name the same columns

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-311.git"
    if not Path("/content/ds-311").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-311"], check=True)
    os.chdir("/content/ds-311/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import gzip
import json
import re
import shutil
import pandas as pd
import yaml
import bayyin as B

out = B.art("d4_4")
src = out / "corrected_src"
src.mkdir(parents=True, exist_ok=True)
for f in (B.DATA / "sources").iterdir():
    shutil.copy(f, src / f.name)
df = pd.read_csv(B.DATA / "sources" / "complaints_extract.csv.gz", dtype=str, keep_default_na=False)
batch = (df.channel == "app") & df.received_date.str.startswith("2026-08")
df.loc[batch, "amount_sar"] = (pd.to_numeric(df.loc[batch, "amount_sar"]) / 100).map(lambda v: f"{v:.2f}")
with gzip.open(src / "complaints_extract.csv.gz", "wt", encoding="utf-8", newline="") as f:
    df.to_csv(f, index=False)
r = subprocess.run([sys.executable, "pipeline/run.py", "--raw", str(src), "--regions", str(src / "regions.csv"),
                    "--out", str(out / "run"), "--run-id", "2026-09-corrected"], capture_output=True, text=True)
print(r.stdout.strip(), r.stderr[-300:])
LOG = json.loads((out / "run" / "run_log.json").read_text(encoding="utf-8"))
AGREE = pd.read_csv(B.REF / "agreement_r1_r2.csv", index_col=0)
print(json.dumps({k: LOG[k] for k in ("run_id", "extract_date", "code_version", "steps", "quarantined", "contract")}, indent=1)[:1800])

## 1 · What a user of the table needs to know (4')

The datasheet questions for this part ask: what preprocessing was done, what was removed and why,
how the labels were made and how reliable they are, and whether the raw data was kept.

✏️ **Exercise** · List the questions an analyst would ask before computing a rate from this table,
at least five, in `QUESTIONS`.

In [ ]:
try:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")
except NotImplementedError:
    QUESTIONS = []
B.check(len(QUESTIONS) >= 5, f"{len(QUESTIONS)} questions")

## 2 · Fill the section from the run log and rule results (9')

Never type a number: take each from `LOG`. The rows a step removed are its `rows_in` minus its
`rows_out`.

✏️ **Exercise** · Complete `facts`, the numbers the section uses, then `section_prep(facts)`.

In [ ]:
steps_ = {s["step"]: s for s in LOG["steps"]}
try:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")
except NotImplementedError:
    facts = {}


def section_prep(f: dict) -> str:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    prep_md = section_prep(facts)
    print(prep_md)
except (NotImplementedError, KeyError):
    prep_md = ""

## 3 · Agreement by variety, and the known limits (7')

The category was labelled under guideline v1.1. Report agreement where it is weakest, with the
variety, not only the overall figure.

✏️ **Exercise** · Write `section_labels()` from `AGREE` (round 2): the overall alpha, the four
varieties, the weakest category; then at least three known limits.

In [ ]:
print(AGREE.to_string())


def section_labels() -> str:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    labels_md = section_labels()
    print(labels_md)
except NotImplementedError:
    labels_md = ""
section = prep_md + "\n" + labels_md
(out / "datasheet_preparation.md").write_text(section, encoding="utf-8")

## 4 · Every number traced to the run (5')

A datasheet with one wrong number loses the reader's trust in all of them. Check mechanically: every
number in the section must appear in the run log (or be the rows a step removed), the agreement
report or the contract.

✏️ **Exercise** · Write `untraced(text)`: find each number in the text and return those that are in
none of the sources.

In [ ]:
def numbers_in(obj) -> set[str]:
    s = json.dumps(obj, ensure_ascii=False) if not isinstance(obj, str) else obj
    return {n.replace(",", "") for n in re.findall(r"\d[\d,]*(?:\.\d+)?", s)}


SOURCES = numbers_in(LOG) | numbers_in(AGREE.to_csv()) | numbers_in((Path("pipeline") / "contract.odcs.yaml").read_text(encoding="utf-8"))
SOURCES |= {"1.1", "300", "3"}  # the guideline version and the pilot's design, from Day 1
SOURCES |= {str(s["rows_in"] - s["rows_out"]) for s in LOG["steps"] if "rows_in" in s}  # rows each step removed


def untraced(text: str) -> list[str]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    missing = untraced(section)
    print("numbers not found in any source:", missing)
    B.check(section != "\n" and not missing, "every number in the section is traced to the run, the agreement report or the contract")
except NotImplementedError:
    print("untraced() not written yet")

## ✱ Stretch · The contract and the datasheet name the same columns

The datasheet should describe each published column; the contract already does. Compare them.

In [ ]:
try:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")
except NotImplementedError:
    print("stretch not done")

## Checkpoint

1. Why take every number from the run log instead of typing it?
   <details><summary>Answer</summary>A typed number goes stale on the next run; a number taken from the log can be regenerated and checked.</details>
2. Why report agreement by variety and not only overall?
   <details><summary>Answer</summary>The overall alpha hides that mixed text is the weakest: an analyst using mixed-text complaints needs that figure.</details>
3. The section says complaints without a region were quarantined. What does that do to the rates?
   <details><summary>Answer</summary>Regional rates slightly understate the total; the limit says so, and the quarantined rows can be counted.</details>

**Carry forward.** `artifacts/d4_4/datasheet_preparation.md`: the capstone's README has the same section
for Al-Ghadaf.